# 🚀 CHẠY AI SERVER (`Qwen3-8B` + `bge-m3`) TRÊN GOOGLE COLAB CHO BACKEND NESTJS
**Công dụng:** Kéo mô hình `Qwen3-8B` đã được bạn Fine-tune và lưu trên **Hugging Face Hub** về chạy trực tiếp trên GPU Tesla T4 (16GB VRAM) của Google Colab, đồng thời mở đường truyền HTTPS công khai qua **Cloudflare Tunnel** (miễn phí, không cần đăng ký tài khoản).

**Ưu điểm:** Laptop 8GB RAM + RTX 3050 Ti của bạn **không tốn 1MB RAM/VRAM hay ổ cứng nào cho AI**, chỉ cần copy đường link URL dán vào file `code/backend/.env`!

In [ ]:
# BƯỚC 1: Cài đặt zstd, Ollama & Cloudflared trên máy ảo Google Colab
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Đã cài đặt xong Ollama và Cloudflared!")

In [ ]:
# BƯỚC 2: Khởi động Ollama Server và Tải mô hình Qwen3-8B từ Hugging Face Hub + bge-m3
import subprocess, time, os

# Điền tên repo Hugging Face chứa mô hình GGUF bạn đã train ở Notebook 1
# Nếu bạn chưa kịp push repo riêng, mặc định sẽ chạy bản qwen3:8b chuẩn để bạn test ngay lập tức!
HF_GGUF_MODEL = "hf.co/tranbateo/qwen3-8b-ai-tutor-vn:Q4_K_M" # Hoặc dùng "qwen3:8b" nếu muốn test bản gốc
EMBEDDING_MODEL = "bge-m3"

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print(f"⏳ Đang tải mô hình LLM ({HF_GGUF_MODEL}) và Embedding ({EMBEDDING_MODEL}) vào GPU Colab...")
!ollama pull {HF_GGUF_MODEL} && ollama cp {HF_GGUF_MODEL} qwen3-8b-ai-tutor-vn || (ollama pull qwen3:8b && ollama cp qwen3:8b qwen3-8b-ai-tutor-vn)
!ollama pull {EMBEDDING_MODEL}
print("✅ Đã nạp xong cả 2 mô hình vào VRAM của Google Colab!")

In [ ]:
# BƯỚC 3: Mở cổng HTTPS công khai và In cấu hình .env cho Backend NestJS
import re, subprocess, time

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 75)
    print("🎉 AI SERVER CỦA BẠN ĐÃ SẴN SÀNG TRÊN GOOGLE COLAB!")
    print("👉 Hãy copy 4 dòng dưới đây dán vào file code/backend/.env trên laptop:")
    print("-" * 75)
    print(f'AI_PROVIDER="OLLAMA"')
    print(f'OLLAMA_BASE_URL="{public_url}"')
    print(f'OLLAMA_MODEL="qwen3-8b-ai-tutor-vn"')
    print(f'OLLAMA_EMBEDDING_MODEL="bge-m3"')
    print("=" * 75)
else:
    print("⚠️ Không lấy được URL tự động, vui lòng chạy lại ô này.")